# Contrastive Probe Inference

Runs the spatial-grounding probe: for each cached scene whose instruction
contains a swappable spatial term, the original and term-swapped instructions
are both passed to OpenVLA against the same frame, and both predictions are
logged with `pair_id`, `role`, `frame`, and `scene_id`, the schema required by
the export notebook.

Two-frame protocol: every pair is predicted against the initial frame, as
before. `placement_relation` pairs additionally get a second prediction
against the grasp frame (the step where the source episode's gripper first
closes), since both variants demand the same initial reach and only diverge
once the object is in hand; the initial-frame prediction for those pairs is
kept as the expected-null baseline. A `placement_relation` scene without a
detected grasp only gets the initial-frame prediction. See
`docs/PROBE_AND_ANALYSIS.md` for the full two-frame protocol and its
inferential caveat.

Every prediction also carries the scene's `category` and `feasible_both`, so the
analysis notebook can stratify (primary target: `referent_selection` pairs judged
feasible on both sides). All categories are logged; a per-category pair-count
summary is printed before the run.

Predictions are written to `probe_predictions_v3.csv`, which adds the `frame`
column; a prior `probe_predictions_v2.csv` (initial frame only) is migrated in
automatically with `frame='initial'` so already-collected predictions are not
re-run.

**Colab GPU:** prefer L4 or A100. Requires notebook 01's install and restart to
have been done in this session, and notebook 02's cache to exist on Drive
(re-run notebook 02's `cache_records` cell first if the manifest predates
grasp-frame extraction, or grasp frames will be unavailable here).


## 1. Mount Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
# v2: adds category and feasible_both columns, initial frame only.
# v3: adds the frame column ('initial' | 'grasp'); v2 rows are migrated in
# with frame='initial' rather than re-predicted (see the migration cell below).
PROBE_CSV_V2 = '/content/drive/MyDrive/openvla_cache/probe_predictions_v2.csv'
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/probe_predictions_v3.csv'
print('cache ->', CACHE_DIR)
print('log   ->', PROBE_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache -> /content/drive/MyDrive/openvla_cache/bridge_multiobj
log   -> /content/drive/MyDrive/openvla_cache/probe_predictions_v3.csv


## 2. Import the code

Imports `model.py` and `data.py` from the Drive-synced repository. Mount Drive
in section 1 first. The default path is `/content/drive/Othercomputers/My MacBook Pro/ECS8056`; change
`REPO_DIR` in the next cell if the folder lives elsewhere on Drive.


In [2]:
import sys, os, importlib

# Drive-synced repository (mount Drive first). Change if the folder path differs.
REPO_DIR = '/content/drive/Othercomputers/My MacBook Pro/ECS8056'


def find_repo_dir(anchor):
    """Return the directory holding `anchor` from known Drive paths only."""
    candidates = []
    if REPO_DIR:
        candidates.append(REPO_DIR)
    candidates.extend([
        '/content/drive/Othercomputers/My MacBook Pro/ECS8056',
        '/content/drive/MyDrive/ECS8056',
        '/content/ECS8056',
    ])
    seen = set()
    for d in candidates:
        d = os.path.abspath(d)
        if d in seen:
            continue
        seen.add(d)
        if os.path.isfile(os.path.join(d, anchor)):
            return d
    return None


module_dir = find_repo_dir('model.py')
if module_dir is None:
    raise FileNotFoundError(
        f"model.py not found on Google Drive. Mount Drive in section 1, confirm ECS8056 has finished syncing, then set REPO_DIR to the folder that contains model.py (tried REPO_DIR={REPO_DIR!r}). Also check with:\n  !ls /content/drive/Othercomputers/My MacBook Pro/ECS8056")
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for m in ('model', 'data'):
    sys.modules.pop(m, None)
importlib.invalidate_caches()

from model import load_openvla, predict_action, run_metadata, append_prediction_log
from data import load_manifest
print('imported model.py and data.py from', module_dir)


imported model.py and data.py from /content/drive/Othercomputers/My MacBook Pro/ECS8056


## 3. Load OpenVLA-7B


In [3]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-80GB (sm_80, 79.3 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[load_openvla] Loaded. GPU memory allocated: 4.08 GB
{'gpu_name': 'NVIDIA A100-SXM4-80GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.11.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.50.0'}


## 4. Generate contrastive pairs from the manifest

Minimal-pair construction by antonym swap: a scene qualifies when its
instruction contains exactly one swappable spatial term, so the pair differs
in that term alone. Instructions with zero or multiple swappable terms are
excluded (a multi-term swap would change more than one relation and break the
minimal-pair property). Role `a` is always the original instruction; role `b`
is the swapped variant.


In [4]:
import re

ANTONYM_PAIRS = [
    ('in front of', 'behind'),
    ('closer to',   'farther from'),
    ('nearer to',   'farther from'),
    ('leftmost',    'rightmost'),
    ('nearest',     'farthest'),
    ('left',        'right'),
    ('top',         'bottom'),
    ('front',       'back'),   
]

SWAP = {}
for a, b in ANTONYM_PAIRS:
    SWAP.setdefault(a, b)
    SWAP.setdefault(b, a)

_KEYS = sorted(SWAP, key=len, reverse=True)
_SWAP_RE = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in _KEYS) + r')\b',
    re.IGNORECASE,
)

def make_pair(instruction: str):
    """Return (term, swapped) if exactly one swappable spatial phrase occurs."""
    hits = _SWAP_RE.findall(instruction)
    if len(hits) != 1:
        return None
    term = hits[0].lower()
    swapped = _SWAP_RE.sub(lambda m: SWAP[m.group(0).lower()], instruction)
    return term, swapped

rows = load_manifest(CACHE_DIR)
probe_set = []
for row in rows:
    made = make_pair(row['instruction'])
    if made is None:
        continue
    term, swapped = made
    # category_manual overrides the heuristic category when a scene has been
    # manually reviewed; downstream code follows the same fallback.
    category = row.get('category_manual') or row.get('category', 'other')
    grasp_rel = row.get('grasp_image_path', '')
    probe_set.append({
        'scene_id': row['episode_index'],
        'pair_id': f"ep{int(row['episode_index']):06d}_{term}",
        'spatial_term': term,
        # Stratification fields carried through to the prediction log.
        'category': category,
        'feasible_both': row.get('feasible_both', 'unreviewed'),
        'image_path': os.path.join(CACHE_DIR, row['image_path']),
        'grasp_image_path': os.path.join(CACHE_DIR, grasp_rel) if grasp_rel else '',
        'instr_a': row['instruction'],
        'instr_b': swapped,
    })

print(f'{len(rows)} cached scenes -> {len(probe_set)} probe pairs')

# All categories are probed; print pair counts so the referent_selection yield
# (the primary analysis target) is visible before committing GPU time.
from collections import Counter
cat_counts = Counter(p['category'] for p in probe_set)
print('pairs per category:', dict(cat_counts))
feas = Counter(p['feasible_both'] for p in probe_set
               if p['category'] == 'referent_selection')
print('referent_selection feasible_both:', dict(feas))

# Grasp-frame availability drives whether a placement_relation pair also gets
# a grasp-frame prediction (see the two-frame protocol above).
placement_pairs = [p for p in probe_set if p['category'] == 'placement_relation']
placement_with_grasp = sum(1 for p in placement_pairs if p['grasp_image_path'])
print(f'placement_relation pairs: {len(placement_pairs)}, '
      f'{placement_with_grasp} with a grasp frame available')
if placement_pairs and placement_with_grasp == 0:
    print('no grasp frames found; re-run notebook 02 cache_records to add them '
          'before grasp-frame predictions can be logged.')

for p in probe_set[:5]:
    print(f"[{p['pair_id']}] ({p['category']})")
    print('  a:', p['instr_a'])
    print('  b:', p['instr_b'])

2239 cached scenes -> 776 probe pairs
pairs per category: {'placement_relation': 375, 'referent_selection': 401}
referent_selection feasible_both: {'no': 7, 'yes': 15, 'unreviewed': 379}
placement_relation pairs: 375, 316 with a grasp frame available
[ep000000_left] (placement_relation)
  a: Place the can to the left of the pot.
  b: Place the can to the right of the pot.
[ep000002_front] (referent_selection)
  a: Slide the cloth diagonally to the front of the spoon
  b: Slide the cloth diagonally to the back of the spoon
[ep000004_right] (referent_selection)
  a: Move the kadai and place it at the right edge of the table.
  b: Move the kadai and place it at the left edge of the table.
[ep000014_top] (referent_selection)
  a: Move the Orange cloth towards the top of the table
  b: Move the Orange cloth towards the bottom of the table
[ep000017_in front of] (placement_relation)
  a: Move the colander in front of the red spoon
  b: Move the colander behind the red spoon


## 5. Run the probe

Two deterministic predictions per (pair, frame) (same frame, both
instructions), each logged with the pairing columns. `sample_idx` is fixed at
0 under the deterministic decoding strategy; the column exists so the schema
does not change if repeated sampling or paraphrase variants are added later.

Restart-safe: `(pair_id, frame)` combinations already present in the log are
skipped, so an interrupted run resumes where it stopped. The next cell
migrates any existing `probe_predictions_v2.csv` into `PROBE_CSV` first (with
`frame='initial'`), so already-collected initial-frame predictions are not
re-run on GPU.


In [ ]:
import pandas as pd

if os.path.exists(PROBE_CSV):
    print(f'{PROBE_CSV} already exists; skipping migration')
elif os.path.exists(PROBE_CSV_V2):
    prev = pd.read_csv(PROBE_CSV_V2)
    prev.insert(list(prev.columns).index('role') + 1, 'frame', 'initial')
    prev.to_csv(PROBE_CSV, index=False)
    print(f'migrated {len(prev)} rows from {PROBE_CSV_V2} -> {PROBE_CSV} '
          "(frame='initial')")
else:
    print(f'no existing log found at {PROBE_CSV_V2}; starting fresh at {PROBE_CSV}')

migrated 144 rows from /content/drive/MyDrive/openvla_cache/probe_predictions_v2.csv -> /content/drive/MyDrive/openvla_cache/probe_predictions_v3.csv (frame='initial')


In [6]:
import csv
import numpy as np
from PIL import Image

done = set()
if os.path.exists(PROBE_CSV):
    with open(PROBE_CSV, newline='') as f:
        for r in csv.DictReader(f):
            done.add((r['pair_id'], r.get('frame') or 'initial'))
    print(f'resuming: {len(done)} (pair_id, frame) combinations already logged')

for i, p in enumerate(probe_set):
    frames_to_run = ['initial']
    if p['category'] == 'placement_relation' and p['grasp_image_path']:
        frames_to_run.append('grasp')

    for frame in frames_to_run:
        if (p['pair_id'], frame) in done:
            continue
        img_path = p['image_path'] if frame == 'initial' else p['grasp_image_path']
        image = Image.open(img_path)
        for role, instr in (('a', p['instr_a']), ('b', p['instr_b'])):
            action = predict_action(processor, vla, image, instr, compute_dtype)
            append_prediction_log(
                PROBE_CSV, action, instr, meta,
                scene_id=p['scene_id'],
                pair_id=p['pair_id'],
                role=role,
                frame=frame,
                spatial_term=p['spatial_term'],
                category=p['category'],
                feasible_both=p['feasible_both'],
                sample_idx=0,
            )
    if (i + 1) % 10 == 0:
        print(f'{i + 1}/{len(probe_set)} pairs done')

print('probe complete ->', PROBE_CSV)

resuming: 72 (pair_id, frame) combinations already logged


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


10/776 pairs done
20/776 pairs done
30/776 pairs done
40/776 pairs done
50/776 pairs done
60/776 pairs done
70/776 pairs done
80/776 pairs done
90/776 pairs done
100/776 pairs done
110/776 pairs done
120/776 pairs done
130/776 pairs done
140/776 pairs done
150/776 pairs done
160/776 pairs done
170/776 pairs done
180/776 pairs done
190/776 pairs done
200/776 pairs done
210/776 pairs done
220/776 pairs done
230/776 pairs done
240/776 pairs done
250/776 pairs done
260/776 pairs done
270/776 pairs done
280/776 pairs done
290/776 pairs done
300/776 pairs done
310/776 pairs done
320/776 pairs done
330/776 pairs done
340/776 pairs done
350/776 pairs done
360/776 pairs done
370/776 pairs done
380/776 pairs done
390/776 pairs done
400/776 pairs done
410/776 pairs done
420/776 pairs done
430/776 pairs done
440/776 pairs done
450/776 pairs done
460/776 pairs done
470/776 pairs done
480/776 pairs done
490/776 pairs done
500/776 pairs done
510/776 pairs done
520/776 pairs done
530/776 pairs done
54

## 6. Quick directional read

A sanity read of the x-axis sign-flip rate for left/right pairs at the
initial frame, plus a separate read for grasp-frame `placement_relation`
predictions, before the export/pilot notebook runs. The formal metrics,
ground-truth validation, and frame checks belong to the next notebook.


In [7]:
import pandas as pd
log = pd.read_csv(PROBE_CSV)
print('frame counts:', dict(log['frame'].value_counts()))

init = log[log['frame'] == 'initial']
lr = init[init['spatial_term'].isin(['left', 'right'])]
wide = lr.pivot_table(index='pair_id', columns='role', values='a0')
flips = (wide['a'] * wide['b'] < 0)
print(f"[initial frame] left/right pairs: {len(wide)} | dx sign flips: {flips.sum()} "
      f"({flips.mean():.1%})")

grasp = log[(log['frame'] == 'grasp') & (log['category'] == 'placement_relation')]
if len(grasp):
    gwide = grasp.pivot_table(index='pair_id', columns='role', values='a0')
    gflips = (gwide['a'] * gwide['b'] < 0)
    print(f"[grasp frame]   placement pairs: {len(gwide)} | dx sign flips: {gflips.sum()} "
          f"({gflips.mean():.1%})")
else:
    print('[grasp frame]   no placement_relation predictions logged yet')
wide.head(8)

frame counts: {'initial': np.int64(1594), 'grasp': np.int64(632)}
[initial frame] left/right pairs: 401 | dx sign flips: 81 (20.2%)
[grasp frame]   placement pairs: 316 | dx sign flips: 78 (24.7%)


role,a,b
pair_id,,
ep000000_left,-0.002669,-0.002669
ep000004_right,-0.014077,-0.012958
ep000027_left,-0.002669,-0.002669
ep000031_left,-0.001327,-0.000209
ep000033_left,-0.004011,-0.004011
ep000036_right,-0.003788,-0.000209
ep000037_right,-0.001327,-0.001327
ep000043_left,-0.002446,-0.002446
